In [ ]:
# ============================================
# Import Required Libraries
# ============================================

import urllib.request
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# ============================================
# Load Haar Cascade
# ============================================

classifier = cv2.CascadeClassifier(
    '/Users/nexusop03/Desktop/AIML_Project/Emotion_Detection_Project/haarcascade_frontalface_default.xml'
)

print("Haar Cascade Loaded Successfully.")

# ============================================
# Load Emotion Detection Model
# ============================================

model = load_model(
    "/Users/nexusop03/Desktop/AIML_Project/Emotion_Detection_Project/emotion_model.keras"
)

print("Emotion Model Loaded Successfully.")

# ============================================
# IP Webcam URL
# ============================================

URL = "http://192.168.1.9:8080/shot.jpg"

print("IP Webcam Connected")

# ============================================
# Emotion Labels
# ============================================

def get_pred_label(pred):

    labels = [
        "Angry",
        "Happy",
        "Neutral",
        "Sad",
        "Surprised"
    ]

    return labels[pred]

# ============================================
# Image Preprocessing
# ============================================

def preprocess(img):

    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    img = cv2.resize(img, (48,48))

    img = img.astype("float32")

    img = img / 255.0

    img = img.reshape(1,48,48,1)

    return img

# ============================================
# Start Emotion Detection
# ============================================

ret = True

while ret:

    img_url = urllib.request.urlopen(URL)

    image = np.array(bytearray(img_url.read()), np.uint8)

    frame = cv2.imdecode(image, -1)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = classifier.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x,y,w,h) in faces:

        face = frame[y:y+h, x:x+w]

        prediction = model.predict(preprocess(face), verbose=0)

        label = np.argmax(prediction)

        emotion = get_pred_label(label)

        confidence = np.max(prediction) * 100

        cv2.rectangle(
            frame,
            (x,y),
            (x+w,y+h),
            (255,0,0),
            2
        )

        cv2.putText(
            frame,
            f"{emotion} {confidence:.1f}%",
            (x,y-10),
            cv2.FONT_HERSHEY_COMPLEX,
            1,
            (255,0,0),
            2
        )

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) == ord("q"):
        break
# ============================================
# Close Window
# ============================================

cv2.destroyAllWindows()

print("Emotion Detection Closed Successfully.")

Haar Cascade Loaded Successfully.
Emotion Model Loaded Successfully.
IP Webcam Connected
